In [ ]:
import os
import sys
sys.path.append('../src')

import pandas as pd

import numpy as np
from scipy.ndimage import center_of_mass
from scipy.optimize import curve_fit
from scipy.fft import fft2, ifft2, fftshift
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse

import swifter

from data_mining.image.common import *

# 配置swifter参数以优化性能
swifter.set_defaults(
    npartitions=os.cpu_count(),  # 分区数量，根据CPU核心数调整
    dask_threshold=100,  # 数据量阈值，超过此数量使用dask
    disable_cache=False,  # 启用缓存
    progress_bar=True  # 显示进度条
)

# Single Image Analysis

In [ ]:
pupil_imgs = pd.DataFrame(
    ['../data/ao/20260115/光瞳开环.png', '../data/ao/20260115/光瞳闭环.png'],
    columns=['path'],
    index=['开环', '闭环']
)

axis_imgs = pd.DataFrame(
    ['../data/ao/20260115/光轴开环.bmp', '../data/ao/20260115/光轴闭环.bmp'],
    columns=['path'],
    index=['开环', '闭环']
)

# 图像分析

## 载入与过滤

In [ ]:
FOCAL_CAM_PIXEL = 2.9
AXIS_CAM_PIXEL = 5.5

In [ ]:
axis_imgs['img_array'] = axis_imgs['path'].apply(read_tiff_to_numpy)
pupil_imgs['img_array'] = pupil_imgs['path'].apply(read_tiff_to_numpy)

In [ ]:
def process_image_data(df, denoise_method='median'):
    """
    处理图像数据：去暗场和计算强度
    
    Args:
        df (pd.DataFrame): 包含img_array列的DataFrame
        denoise_method (str): 去暗场方法，'median' 或 'min'
    
    Returns:
        pd.DataFrame: 处理后的DataFrame
    """
    df = df.copy()
    
    # 去暗场
    if denoise_method == 'median':
        df['black'] = df['img_array'].swifter.apply(lambda x: np.median(x))
    elif denoise_method == 'min':
        df['black'] = df['img_array'].swifter.apply(lambda x: np.min(x))
    elif denoise_method == '1_e':
        df['black'] = df['img_array'].swifter.apply(lambda x: np.max(x)/np.e)
    
    # 去噪后的图像
    black_threshold = df['black'].max()
    df['denoise_img_array'] = df.apply(lambda x: np.where(x['img_array'] > black_threshold, x['img_array'] - black_threshold, 0), axis=1)
    
    # 计算总强度
    df['intensity'] = df['denoise_img_array'].swifter.apply(np.sum)
    
    return df

pupil_beam = process_image_data(pupil_imgs)
axis_beam = process_image_data(axis_imgs, denoise_method='1_e')

## 提取一阶矩、二阶矩

In [ ]:
# 提取一阶矩、二阶矩
def d4sigma(img : np.ndarray, pixel_size_um=1.0):
    """
    计算图像的 D4σ 直径（一阶矩和二阶矩）
    
    Args:
        img (np.ndarray): 输入图像（2D数组）
        pixel_size_um (float): 像素尺寸（微米）
    
    Returns:
        tuple: (中心x, 中心y, D4σ_x, D4σ_y)
    """
    total = img.sum()
    cy, cx = center_of_mass(img)
    h, w = img.shape
    y, x = np.mgrid[0:h, 0:w]
    
    # 二阶中心矩（光强加权）
    mu_xx = np.sum((x - cx)**2 * img) / total  # σ_x²
    mu_yy = np.sum((y - cy)**2 * img) / total  # σ_y²
    Dx = 4 * np.sqrt(max(mu_xx, 0)) * pixel_size_um
    Dy = 4 * np.sqrt(max(mu_yy, 0)) * pixel_size_um
    
    return {
        'center_x': float(cx),
        'center_y': float(cy),
        'D_x': float(Dx),
        'D_y': float(Dy),
        'center_intensity': float(img[int(cy), int(cx)]),
    }

def d4sigma_feature_extract(df: pd.DataFrame, pixel_size_um=1.0):
    d4sigma_features = df.swifter.apply(lambda x: d4sigma(x['denoise_img_array'], pixel_size_um), axis=1, result_type='expand')
    d4sigma_features['avg_sigma2'] = np.sqrt(d4sigma_features['D_x'] * d4sigma_features['D_y'])
    return pd.merge(df, d4sigma_features, left_index=True, right_index=True)

valid_axis_beam = d4sigma_feature_extract(axis_beam, AXIS_CAM_PIXEL)
valid_pupil_beam = d4sigma_feature_extract(pupil_beam, FOCAL_CAM_PIXEL)

valid_axis_beam['avg_sigma2']

In [ ]:
open_axis_img = valid_axis_beam['img_array'].iloc[0]
close_axis_img = valid_axis_beam['img_array'].iloc[1]

fig,( ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ax1.imshow(open_axis_img, cmap='gray')
ax1.scatter(valid_axis_beam['center_x'].iloc[0], valid_axis_beam['center_y'].iloc[0], c='red', marker='x')
ax2.imshow(close_axis_img, cmap='gray')
ax2.scatter(valid_axis_beam['center_x'].iloc[1], valid_axis_beam['center_y'].iloc[1], c='red', marker='x')


In [ ]:
# PIB 占比
def pib_ratio(img, center, r=5.0):
    """
    计算 PIB 占比
    
    Args:
        img (np.ndarray): 输入图像（2D数组）
        center (tuple): 中心坐标 (x, y)
        r (float): 半径（默认5.0）
    
    Returns:
        float: PIB 占比
    """
    cx, cy = center
    h, w = img.shape
    y, x = np.mgrid[0:h, 0:w]
    mask = (x - cx)**2 + (y - cy)**2 <= r**2
    pib_intensity = img[mask].sum()
    total_intensity = img.sum()
    return pib_intensity / total_intensity

pib_ratio = valid_axis_beam.apply(lambda x: pib_ratio(x['img_array'], (x['center_x'], x['center_y'])), axis=1)
valid_axis_beam['pib_ratio'] = pib_ratio

pib_ratio

## 高斯拟合

$$
I(x) = \frac{A}{\sqrt{2\pi}\sigma} e^{- \frac{(x - \mu)^2}{2\sigma^2}} + offset
$$

$$
w = \frac{1}{\sqrt{2\pi}\sigma}
$$

其中，
- $ A $：归一化系数，
- $ \mu $：光强最大点（中心），
- $ \sigma $：光强衰减宽度，
- $ offset $：常数偏移项。

In [ ]:
# 高斯拟合
def gaussian(x, mu, sigma, A, b):
    """
    Define the Gaussian function.

    Args:
        x (np.ndarray): Input x values.
        A (float): Amplitude of the Gaussian.
        mu (float): Mean of the Gaussian.
        sigma (float): Standard deviation of the Gaussian.

    Returns:
        np.ndarray: Output values of the Gaussian function.
    """
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) + b

def fitting_gaussian(data):
    """
    Fit a Gaussian function to a given data series.
    Args:
        data (np.ndarray): The data series to fit a Gaussian function.
    Returns:
        tuple: A tuple containing the fitted parameters (A, b, mu, sigma) and the fitted curve.
    """
    x_data = np.arange(len(data))
    initial_guess = [np.argmax(data), 10, np.max(data), 0]
    try:
        (mu, sigma, A, b), covariance = curve_fit(gaussian, x_data, data, p0=initial_guess)
    except RuntimeError:
        return (np.nan, np.nan, np.nan, np.nan), np.nan

    return (mu, sigma, A, b), covariance

def calculate_diameter(sigma):
    diameter = 2 * sigma
    return diameter

def calculate_xy_diameters(image, center_x, center_y, pix_size=1.0):
    """
    Calculate the diameters at y = 1/e + b in x and y directions.

    Args:
        image (np.ndarray): The input image array.
        centroid (tuple): The (y, x) coordinates of the centroid.

    Returns:
        tuple: A tuple containing the x-direction diameter and y-direction diameter.
    """
    # Extract data for x and y directions
    y_data = image[:, int(center_x)]
    x_data = image[int(center_y), :]

    # Calculate diameters
    (mu, sigma, A, b), conv = fitting_gaussian(x_data)
    x_diameter = calculate_diameter(sigma)
    (mu, sigma, A, b), conv = fitting_gaussian(y_data)
    y_diameter = calculate_diameter(sigma)

    return {'gaussian_dia_x(um)': x_diameter*pix_size, 'gaussian_dia_y(um)': y_diameter*pix_size}

guassian_dia = valid_axis_beam.swifter.apply(
    lambda x: calculate_xy_diameters(x['img_array'], x['center_x'], x['center_y'], pix_size=AXIS_CAM_PIXEL),
    axis=1, result_type='expand'
)

valid_axis_beam = pd.merge(valid_axis_beam, guassian_dia, left_index=True, right_index=True)
guassian_dia

## 椭圆拟合

In [ ]:
# 椭圆拟合
def convert_to_cv(float_image) -> np.ndarray:
    """
    将图像转换为OpenCV兼容的uint8格式
    """
    assert isinstance(float_image, np.ndarray), f"{float_image} must be a numpy array, but got {type(float_image)}"
    image_min = np.min(float_image)
    image_max = np.max(float_image)

    normalized_image = (float_image - image_min) / (image_max - image_min) * 255
    uint8_image = normalized_image.astype(np.uint8)
    return uint8_image

def to_binary_image(uint8_imag, threshold=None, block_size=15, C=2, otsu_blur_ksize=3):
    # 计算噪声阈值
    # 3. 应用自适应阈值
    if threshold == "gaussian":
        binary = cv2.adaptiveThreshold(
            uint8_imag,
            maxValue=255,
            adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            thresholdType=cv2.THRESH_BINARY,
            blockSize=block_size,
            C=C,
        )
    elif threshold == "mean":
        binary = cv2.adaptiveThreshold(
            uint8_imag,
            maxValue=255,
            adaptiveMethod=cv2.ADAPTIVE_THRESH_MEAN_C,
            thresholdType=cv2.THRESH_BINARY,
            blockSize=block_size,
            C=C,
        )
    elif threshold == "otsu":
        blurred = cv2.GaussianBlur(uint8_imag, (otsu_blur_ksize, otsu_blur_ksize), 0)
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif isinstance(threshold, (int, float)):
        _, binary = cv2.threshold(uint8_imag, threshold, 255, cv2.THRESH_BINARY)
    else:
        binary = cv2.threshold(uint8_imag, 0, 255, 1)[1]
    
    # 4. 后处理：形态学开运算（去噪点）
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    
    return binary
    
    return 

def ellipse_fit(uint8_image):    
    # 查找轮廓
    try:
        contours, _ = cv2.findContours(uint8_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        assert contours, "No contours found"
        # 找到最大的轮廓
        largest_contour = max(contours, key=cv2.contourArea)
        (ellipse_center_x, ellipse_center_y),(short_axis, long_axis),angle = cv2.fitEllipse(largest_contour)
    except (AssertionError, ValueError):
        return {
            'ellipse_center_x': np.nan,
            'ellipse_center_y': np.nan,
            'short_axis': np.nan,
            'long_axis': np.nan,
            'ellipticity': np.nan,
            'angle': np.nan,
            'uniformity': np.nan
        }
    
    area = cv2.contourArea(largest_contour)
    # 创建掩膜用于后续处理
    mask = np.zeros_like(uint8_image, dtype=np.uint8)
    if area > 100:
        # 将主轮廓内部填充为白色 (255)
        cv2.drawContours(mask, [largest_contour], -1, (255,), thickness=cv2.FILLED)
        # 使用掩膜提取光斑内的所有像素
        mean_val, std_val = cv2.meanStdDev(uint8_image, mask=mask)
        mean_intensity = mean_val[0][0]
        std_intensity = std_val[0][0]
        uniformity = std_intensity / mean_intensity
    else:
        uniformity = np.nan
    
    return {
        'ellipse_center_x': ellipse_center_x,
        'ellipse_center_y': ellipse_center_y,
        'short_axis': short_axis,
        'long_axis': long_axis,
        'ellipticity': long_axis / short_axis,
        'angle': angle,
        'uniformity': uniformity
    }
    
def find_spot_border(binary_image):
    """
    处理光斑图片，计算噪声阈值，去除噪声并拟合包含光斑的圆形。

    参数:
    image (numpy.ndarray): 输入的光斑图片，应为单通道灰度图像。

    返回:
    numpy.ndarray: 去除噪声后的图像。
    tuple: 拟合圆形的圆心坐标 (x, y) 和半径。
    """
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # 找到最大的轮廓
        largest_contour = max(contours, key=cv2.contourArea)
        ((x, y), radius) = cv2.minEnclosingCircle(largest_contour)
    else:
        x, y = np.nan, np.nan
        radius = np.nan

    return {
        'border_x': x,
        'border_y': y,
        'border_radius': radius
    }

def shape_feature_extract(df: pd.DataFrame, denoise):
    """
    提取形状特征，支持GPU加速
    
    参数:
    df: 包含denoise_img_array列的DataFrame
    use_gpu: 是否使用GPU加速
    
    返回:
    包含形状特征的DataFrame
    """
    # 转换图像格式
    uint8_images = df['denoise_img_array']\
        .apply(convert_to_cv)\
        .apply(to_binary_image, threshold=denoise)
    ellipse_features = pd.DataFrame(uint8_images.apply(ellipse_fit).tolist(), index=df.index)
    circle_features = pd.DataFrame(uint8_images.apply(find_spot_border).tolist(), index=df.index)
    
    return pd.merge(df, ellipse_features, left_index=True, right_index=True).merge(circle_features, left_index=True, right_index=True)
    # return pd.merge(df, circle_features, left_index=True, right_index=True)
    
valid_axis_beam = shape_feature_extract(valid_axis_beam, denoise=1)
valid_pupil_beam = shape_feature_extract(valid_pupil_beam, denoise="otsu")


In [ ]:
fig,( ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

open_pupil_beam = valid_pupil_beam['img_array'].iloc[0]
close_pupil_beam = valid_pupil_beam['img_array'].iloc[1]

ax1.imshow(open_pupil_beam, cmap='gray')
ax1.add_patch(Circle((valid_pupil_beam['border_x'].iloc[0], valid_pupil_beam['border_y'].iloc[0]), valid_pupil_beam['border_radius'].iloc[0], color='red', fill=False))
ax2.imshow(close_pupil_beam, cmap='gray')
ax2.add_patch(Circle((valid_pupil_beam['border_x'].iloc[1], valid_pupil_beam['border_y'].iloc[1]), valid_pupil_beam['border_radius'].iloc[1], color='red', fill=False))


In [ ]:
fig,( ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

open_axis_beam = valid_axis_beam['img_array'].iloc[0]
close_axis_beam = valid_axis_beam['img_array'].iloc[1]

ax1.imshow(open_axis_beam, cmap='gray')
ax1.add_patch(Circle((valid_axis_beam['border_x'].iloc[0], valid_axis_beam['border_y'].iloc[0]), valid_axis_beam['border_radius'].iloc[0], color='red', fill=False))
ax1.add_patch(Ellipse((valid_axis_beam['ellipse_center_x'].iloc[0], valid_axis_beam['ellipse_center_y'].iloc[0]), valid_axis_beam['long_axis'].iloc[0], valid_axis_beam['short_axis'].iloc[0], angle=valid_axis_beam['angle'].iloc[0], color='blue', fill=False))
ax2.imshow(close_axis_beam, cmap='gray')
ax2.add_patch(Circle((valid_axis_beam['border_x'].iloc[1], valid_axis_beam['border_y'].iloc[1]), valid_axis_beam['border_radius'].iloc[1], color='red', fill=False))


# 中心平移

In [ ]:
def shift_to_center_fft(image, cx, cy):
    """
    使用傅里叶移位将光斑移到图像中心（无插值，保全信息）
    
    参数:
        image: 2D array
        target_center: (cx, cy) 目标中心，默认为图像几何中心
    
    返回:
        shifted_image: 光斑已居中的图像
    """
    image = np.asarray(image, dtype=float)
    h, w = image.shape
    dx = w//2 - cx
    dy = h//2 - cy
    
    # 傅里叶移位：在频域乘以相位因子
    # 创建频率网格
    u = np.fft.fftfreq(w).reshape(1, -1)
    v = np.fft.fftfreq(h).reshape(-1, 1)

    # 相位因子：exp(-2πi (u*dx + v*dy))
    phase = np.exp(-2j * np.pi * (u * dx + v * dy))

    # 应用移位
    F = np.fft.fft2(image)
    F_shifted = F * phase
    shifted = np.real(np.fft.ifft2(F_shifted))

    # 保留非负强度（数值误差可能导致微小负值）
    shifted = np.clip(shifted, 0, None)
    return np.where(shifted < 1e-3, 0, shifted)

valid_pupil_beam['shifted_img_array'] = valid_pupil_beam.apply(
    lambda x: shift_to_center_fft(x['denoise_img_array'], x['center_x'], x['center_y']), axis=1
)
valid_axis_beam['shifted_img_array'] = valid_axis_beam.apply(
    lambda x: shift_to_center_fft(x['denoise_img_array'], x['center_x'], x['center_y']), axis=1
)

In [ ]:
c = ['ellipticity', 'uniformity','border_radius']
valid_axis_beam['ellipticity'].plot(subplots=True)
valid_pupil_beam[c]

## 环围

In [ ]:
# TODO 环围
def make_coord(img:np.ndarray):
    """
    生成坐标矩阵

    :param img: 强度分布
    :return x, y: 坐标矩阵
    """
    h, w = img.shape
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    return x, y

def radius(intensity, center, energy=0.99):
    """
    以center为圆心，占总能量百分比为energy的圆的半径

    :param intensity: 强度分布
    :param x: x坐标矩阵
    :param y: y坐标矩阵
    :param center: 圆心，坐标，如(0, 0)
    :param energy: 圆内的能量比，默认0.99，取值范围0~1，常用0.5，0.865， 0.99
    :return radius: 圆的半径
    """
    x, y = make_coord(intensity)
    npix = len(x)
    dpix = x[0, 1] - x[0, 0]
    
    x0, y0 = center[0], center[1]

    power_in_circle = np.sum(intensity) * energy
    r = np.sqrt((x - x0) ** 2 + (y - y0) ** 2)
    radius = npix * dpix / 2
    radius_change = npix * dpix / 4

    for i in range(300):
        mask = (np.sign(radius - r) + 1) / 2
        power = np.sum(intensity * mask)

        if power - power_in_circle < -1e-10 * power_in_circle:
            radius += radius_change
        elif power - power_in_circle > 1e-10 * power_in_circle:
            radius -= radius_change
        else:
            break

        radius_change /= 2
        if radius_change < dpix / 50:
            break
        if i == 299:
            return np.nan

    return {'power99_radius':radius}

def calc_radius(x: pd.Series):
    radius_feature = radius(x['denoise_img_array'], center = (x['center_x'], x['center_y']))
    return radius_feature

radius_feature = valid_pupil_beam.swifter.apply(calc_radius ,axis=1, result_type='expand')
valid_pupil_beam = pd.merge(valid_pupil_beam, radius_feature, left_index=True, right_index=True)

radius_feature.plot(subplots=True)

In [ ]:
# TODO zernike 

# 光瞳光轴对齐

In [ ]:
merged_beam = valid_axis_beam.join(valid_pupil_beam, lsuffix='_axis', rsuffix='_pupil')
merged_beam

## strehl

In [ ]:
def calculate_strehl_ratio_with_energy_conservation(
    pupil_img,
    focus_img,
    pixel_size_pupil_um=2.9,
    pixel_size_focus_um=4.5,
    N=1/32,
    f_mm=3_000,
    wavelength_um=1.064,
    background_subtract=False,
    roi_fraction=0.8
):
    """
    基于能量守恒的斯特列尔比计算。
    
    新增特性:
        - 背景扣除
        - 能量归一化（使理想与实际总能量一致）
        - 可选 ROI 避免边缘噪声影响
    
    返回:
        strehl_ratio (float)
        ideal_matched (np.ndarray): 能量匹配后的理想光斑（与 focus_img 同尺寸）
    """

    # ----------------------------
    # 1. 背景扣除（可选但推荐）
    # ----------------------------
    def subtract_background(img):
        # 使用图像边缘区域估计背景（假设中心是光斑）
        h, w = img.shape
        margin = int(min(h, w) * 0.1)
        bg = np.median(img[margin:-margin, margin:-margin])
        return np.maximum(img.astype(np.float64) - bg, 0.0)

    if background_subtract:
        pupil_img = subtract_background(pupil_img)
        focus_img = subtract_background(focus_img)

    # ----------------------------
    # 2. 物理尺度校准
    # ----------------------------
    dx_pupil_phys_um = pixel_size_pupil_um / N  # 关键：除以缩束比 N
    H, W = pupil_img.shape
    Lx_pupil_um = W * dx_pupil_phys_um
    Ly_pupil_um = H * dx_pupil_phys_um

    # ----------------------------
    # 3. 构建理想复振幅（假设相位为0）
    # ----------------------------
    pupil_field = np.sqrt(np.maximum(pupil_img, 0)) + 0j  # 复数类型

    # ----------------------------
    # 4. 计算理想聚焦光斑（透镜后焦面）
    # ----------------------------
    ideal_focus_field = fftshift(fft2(ifftshift(pupil_field)))
    ideal_focus_intensity = np.abs(ideal_focus_field)**2

    # ----------------------------
    # 5. 计算理想光斑物理网格
    # ----------------------------
    f_um = f_mm * 1000.0
    dx_ideal_um = (wavelength_um * f_um) / Lx_pupil_um
    dy_ideal_um = (wavelength_um * f_um) / Ly_pupil_um
    # 理想光斑总物理尺寸
    Lx_ideal_um = W * dx_ideal_um
    Ly_ideal_um = H * dy_ideal_um

    x_ideal = np.linspace(-Lx_ideal_um/2, Lx_ideal_um/2 - dx_ideal_um, W)
    y_ideal = np.linspace(-Ly_ideal_um/2, Ly_ideal_um/2 - dy_ideal_um, H)
    X_ideal, Y_ideal = np.meshgrid(x_ideal, y_ideal)

    # ----------------------------
    # 6. 实际光斑物理网格
    # ----------------------------
    Hf, Wf = focus_img.shape
    x_actual = (np.arange(Wf) - Wf // 2) * pixel_size_focus_um
    y_actual = (np.arange(Hf) - Hf // 2) * pixel_size_focus_um
    X_actual, Y_actual = np.meshgrid(x_actual, y_actual)

    # ----------------------------
    # 7. 插值：理想 → 实际网格
    # ----------------------------
    interp_func = RegularGridInterpolator(
        (y_ideal, x_ideal),
        ideal_focus_intensity,
        method='linear',
        bounds_error=False,
        fill_value=0.0
    )
    points = np.stack([Y_actual.ravel(), X_actual.ravel()], axis=-1)
    ideal_on_actual = interp_func(points).reshape(Hf, Wf)

    # ----------------------------
    # 8. 【关键】能量守恒校准
    # ----------------------------
    # 可选：使用中心 ROI 计算能量，避免边缘噪声
    if roi_fraction < 1.0:
        h_roi = int(Hf * roi_fraction)
        w_roi = int(Wf * roi_fraction)
        y_start = (Hf - h_roi) // 2
        x_start = (Wf - w_roi) // 2
        
        actual_roi = focus_img[y_start:y_start+h_roi, x_start:x_start+w_roi]
        ideal_roi = ideal_on_actual[y_start:y_start+h_roi, x_start:x_start+w_roi]
    else:
        actual_roi = focus_img
        ideal_roi = ideal_on_actual

    total_energy_actual = np.sum(actual_roi)
    total_energy_ideal = np.sum(ideal_roi)

    if total_energy_ideal == 0:
        raise ValueError("理想光斑总能量为零，请检查输入光瞳图像。")

    # 缩放理想光斑，使其总能量 = 实际总能量
    scaling_factor = total_energy_actual / total_energy_ideal
    ideal_energy_matched = ideal_on_actual * scaling_factor

    # ----------------------------
    # 9. 计算斯特列尔比
    # ----------------------------
    peak_actual = np.max(actual_roi)
    peak_ideal = np.max(ideal_energy_matched[
        y_start:y_start+h_roi, x_start:x_start+w_roi
    ]) if roi_fraction < 1.0 else np.max(ideal_energy_matched)

    strehl = peak_actual / (peak_ideal + 1e-12)

    return strehl, ideal_energy_matched

strehl_results_with_scaler = merged_beam.swifter.apply(
    lambda row: calculate_strehl_ratio_with_energy_conservation(row['shifted_img_array_pupil'], row['shifted_img_array_axis']),
    axis=1,
    result_type='expand'
)
strehl_results_with_scaler.columns = ['strehl_ratio', 'ideal_matched']
merged_beam = pd.merge(merged_beam, strehl_results_with_scaler, left_index=True, right_index=True)
strehl_results_with_scaler

## 计算BPP和M²

In [ ]:
def calculate_bpp_from_pupil_and_focal(
    pupil_diameter_mm,
    focal_diameter_mm,
    focal_length_mm = 3e3,
    wavelength_nm=1064.0,
    beam_expansion_ratio=1.0/32,
    pupil_diameter_input_mm=None
):
    """
    使用出瞳和焦斑直径（单位：mm）计算 BPP 和 M²，考虑缩束比
    
    Parameters:
        pupil_diameter_mm: 出瞳 D4σ 直径（毫米）
        focal_diameter_mm: 焦平面 D4σ 直径（毫米）
        focal_length_mm: 透镜焦距（毫米）
        wavelength_nm: 波长（纳米）
        beam_expansion_ratio: 光束扩束/缩束比 (>1 表示扩束, <1 表示缩束)
        pupil_diameter_input_mm: 输入光束直径（毫米），用于计算有效扩束比
    
    Returns:
        dict with results in mm / mm·mrad / mrad
    """
    # 转为半径（mm）
    w_pupil = pupil_diameter_mm / 2.0      # mm
    w_focal = focal_diameter_mm / 2.0      # mm
    f = focal_length_mm                    # mm
    
    # 发散角 θ ≈ w_focal / f （单位：弧度）
    theta_rad = w_focal / f
    theta_mrad = theta_rad * 1000.0        # 转为毫弧度（mrad）
    
    # BPP = w_pupil * θ （单位：mm·rad → 转为 mm·mrad）
    bpp_mm_mrad = w_pupil * theta_mrad
    
    # 如果提供了输入光束直径，计算实际扩束比
    if pupil_diameter_input_mm is not None:
        actual_expansion_ratio = pupil_diameter_mm / pupil_diameter_input_mm
        effective_expansion_ratio = actual_expansion_ratio
    else:
        effective_expansion_ratio = beam_expansion_ratio
        actual_expansion_ratio = beam_expansion_ratio
    
    # 计算理想扩束后的理论BPP
    # bpp_theoretical_input = None
    # if pupil_diameter_input_mm is not None:
    #     w_input = pupil_diameter_input_mm / 2.0
    #     bpp_theoretical_input = w_input * theta_mrad
    
    # 衍射极限 BPP (mm·mrad) = λ(μm) / π
    # wavelength_um = wavelength_nm * 1e-3   # nm → μm
    # bpp_diffraction_mm_mrad = wavelength_um / np.pi
    
    # M2 = bpp_mm_mrad / bpp_diffraction_mm_mrad
    
    # 考虑缩束比的影响
    # M2_corrected = M2 / effective_expansion_ratio if effective_expansion_ratio != 0 else np.nan
    
    return {
        "BPP_mm_mrad": bpp_mm_mrad,
    }
    
bpp_results = merged_beam.swifter.apply(
    lambda row: calculate_bpp_from_pupil_and_focal(row['avg_sigma2_pupil'], row['avg_sigma2_axis']),
    axis=1, result_type='expand')

merged_beam = pd.merge(merged_beam, bpp_results, left_index=True, right_index=True)
bpp_results

# 可视化分析

In [ ]:
import  pygwalker

def filter_pygwalker_supported_columns(df):
    supported_dtypes = ['int64', 'float64', 'int32', 'float32', 'int16', 'float16', 
                       'int8', 'uint8', 'uint16', 'uint32', 'datetime64[ns]', 
                       'category', 'string', 'object']
    
    supported_columns = []

    columns_without_xy = {}
    for c in df.columns:
        columns_without_xy.get(c.replace('_x', '').replace('_y', ''))
    
    for col in df.columns:
        dtype = str(df[col].dtype)
        
        # 检查是否是基本支持的数据类型
        if dtype in supported_dtypes:
            # 对于 object 类型，需要进一步检查是否为复杂对象（如 numpy 数组）
            if dtype == 'object':
                # 检查是否为简单的可序列化对象（如字符串、Path等）
                sample_value = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
                if sample_value is not None:
                    # 检查是否为 numpy 数组或其他复杂对象
                    if isinstance(sample_value, (np.ndarray, tuple, list, dict)):
                        continue  # 跳过复杂对象
                supported_columns.append(col)
            else:
                supported_columns.append(col)
    
    return df[supported_columns]

# 1. 按time列排序merged_beam
merged_beam = merged_beam.sort_values('time')
# 调用函数进行分析
supported_df = filter_pygwalker_supported_columns(merged_beam)

pygwalker.walk(supported_df, appearance='light', theme_key='vega')